# Klasifikasi Sentimen Bi-LSTM — Optimasi Konfigurasi & CV Bebas Kebocoran Data

Notebook ini telah diperbaiki secara keseluruhan sesuai alur penelitian yang direkomendasikan:
1. **Pencarian Konfigurasi Terbaik (Tanpa Cross-Validation)**:
   - Menjalankan 4 eksperimen (PB1 - PB4) menggunakan satu set pembagian data tetap (**Train 80% - Val 10% - Test 10%**).
   - Konfigurasi terbaik dipilih secara otomatis berdasarkan metrik **Macro F1** tertinggi pada set validasi.
2. **Evaluasi Cross-Validation (Hanya untuk Konfigurasi Terbaik)**:
   - Melakukan **5-Fold Cross Validation** khusus untuk konfigurasi terbaik tersebut untuk menguji stabilitas model.
3. **Model Akhir & Pengujian Utama**:
   - Melatih model akhir pada gabungan data latih + validasi (90% data) menggunakan konfigurasi terbaik.
   - Menguji performa final pada **Test Set** (10% data) yang bersih dari intervensi oversampling maupun fitting tokenizer.

In [ ]:
import os
import json
import pickle
import random
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# Pasang imbalanced-learn jika belum tersedia
try:
    from imblearn.over_sampling import RandomOverSampler
except ImportError:
    print("[INFO] Memasang imbalanced-learn...")
    os.system("pip install -q imbalanced-learn")
    from imblearn.over_sampling import RandomOverSampler

warnings.filterwarnings("ignore")

# Parameter Reproduksibilitas
RANDOM_STATE = 42

# Path Dataset (Mendukung Kaggle & Fallback Lokal)
DATA_PATH = "/kaggle/input/datasets/salwafitriyatunnisa/dataset-pdp-sosial-media-final/1. Dataset PDP Sosial Media.xlsx"
if not os.path.exists(DATA_PATH):
    possible_paths = [
        "1. Dataset PDP Sosial Media.xlsx",
        "C:/Users/HP/Downloads/1. Dataset PDP Sosial Media.xlsx",
        r"C:\Users\HP\Downloads\dataset_labeled_sosmed_2_lengkap_manual label_done.xlsx",
        r"C:\Users\HP\Downloads\1. Dataset Berita Daring.xlsx"
    ]
    for p in possible_paths:
        if os.path.exists(p):
            DATA_PATH = p
            print(f"[INFO] Dataset ditemukan di path lokal: {DATA_PATH}")
            break

TEXT_COLUMN = "text"
LABEL_COLUMN = "sentiment"

# Parameter Model
MAX_LEN = 256
VOCAB_SIZE = 30000
EMBED_DIM = 300
HIDDEN_DIM = 256

# Parameter Pelatihan
EPOCHS = 20
PATIENCE = 3
N_SPLITS = 5

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_OUTPUT_DIR = os.path.join("outputs", "bilstm_sosmed", RUN_ID)
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_seed(RANDOM_STATE)
print(f"Output akan disimpan di: {BASE_OUTPUT_DIR}")


In [ ]:
# 1. Muat Data
df = pd.read_excel(DATA_PATH)
df = df[[TEXT_COLUMN, LABEL_COLUMN]].dropna()
print("Original dataset shape:", df.shape)

# 2. Hapus duplikat untuk mencegah kebocoran data
df.drop_duplicates(subset=[TEXT_COLUMN], inplace=True)
df = df.reset_index(drop=True)
print("Deduplicated dataset shape:", df.shape)

# 3. Label Encoding
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df[LABEL_COLUMN] = label_encoder.fit_transform(df[LABEL_COLUMN])

label_names = label_encoder.classes_.tolist()
num_labels = len(label_names)

print("\nLabel Mapping:")
for idx, label in enumerate(label_names):
    print(f"{idx} -> {label}")

label_mapping = {int(i): str(label) for i, label in enumerate(label_names)}
with open(os.path.join(BASE_OUTPUT_DIR, "label_mapping.json"), "w") as f:
    json.dump(label_mapping, f, indent=4)

# 4. Pembagian Data (80% Train, 10% Val, 10% Test)
# Bagi 80-20 terlebih dahulu
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df[LABEL_COLUMN],
    random_state=RANDOM_STATE
)

# Bagi sisa 20% menjadi 10% validasi dan 10% pengujian
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[LABEL_COLUMN],
    random_state=RANDOM_STATE
)

print("\nDataset Splits:")
print("Train Set      :", train_df.shape)
print("Validation Set :", val_df.shape)
print("Test Set       :", test_df.shape)

train_df[LABEL_COLUMN].value_counts().sort_index().to_csv(os.path.join(BASE_OUTPUT_DIR, "train_label_distribution.csv"))
val_df[LABEL_COLUMN].value_counts().sort_index().to_csv(os.path.join(BASE_OUTPUT_DIR, "val_label_distribution.csv"))
test_df[LABEL_COLUMN].value_counts().sort_index().to_csv(os.path.join(BASE_OUTPUT_DIR, "test_label_distribution.csv"))


In [ ]:
def build_model():
    model = Sequential([
        Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),
        Bidirectional(LSTM(HIDDEN_DIM)),
        Dropout(0.3),
        Dense(num_labels, activation="softmax")
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def save_confusion_matrix(y_true, y_pred, save_path, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=label_names, yticklabels=label_names
    )
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

def plot_history(history, save_path, title):
    plt.figure(figsize=(8,5))
    if hasattr(history, 'history'):
        train_loss = history.history['loss']
        val_loss = history.history.get('val_loss', None)
    elif isinstance(history, dict):
        train_loss = history['loss']
        val_loss = history.get('val_loss', None)
    else:
        train_loss = history['loss']
        val_loss = history['val_loss'] if 'val_loss' in history else None
    plt.plot(train_loss, label="Train Loss")
    if val_loss is not None:
        plt.plot(val_loss, label="Val Loss")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()


In [ ]:
def run_single_split_experiment(exp_code, use_ros, batch_size):
    print("="*80)
    print(f"JALANKAN EKSPERIMEN: {exp_code} (ROS={use_ros}, Batch Size={batch_size})")
    print("="*80)
    
    exp_dir = os.path.join(BASE_OUTPUT_DIR, exp_code)
    os.makedirs(exp_dir, exist_ok=True)
    
    # 1. Terapkan ROS hanya pada data latih (jika diaktifkan)
    if use_ros:
        ros = RandomOverSampler(random_state=RANDOM_STATE)
        X_res, y_res = ros.fit_resample(
            train_df[TEXT_COLUMN].values.reshape(-1, 1),
            train_df[LABEL_COLUMN]
        )
        train_text = X_res.flatten()
        train_label = y_res
    else:
        train_text = train_df[TEXT_COLUMN].values
        train_label = train_df[LABEL_COLUMN].values
        
    # 2. Fit Tokenizer hanya pada data latih (tanpa menyentuh val & test)
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
    tokenizer.fit_on_texts(train_text)
    
    # 3. Tokenisasi dan Padding
    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_text), maxlen=MAX_LEN, padding="post", truncating="post")
    X_val_seq = pad_sequences(tokenizer.texts_to_sequences(val_df[TEXT_COLUMN].values), maxlen=MAX_LEN, padding="post", truncating="post")
    
    # 4. Latih Model
    model = build_model()
    early_stop = EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    
    history = model.fit(
        X_train_seq, train_label,
        validation_data=(X_val_seq, val_df[LABEL_COLUMN].values),
        epochs=EPOCHS,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )
    
    # Simpan plot loss & history
    pd.DataFrame(history.history).to_csv(os.path.join(exp_dir, "history_training.csv"), index=False)
    plot_history(history, os.path.join(exp_dir, "loss_curve.png"), f"Loss Curve - {exp_code}")
    
    # 5. Evaluasi pada Set Validasi untuk pencarian parameter
    val_preds = np.argmax(model.predict(X_val_seq), axis=1)
    val_accuracy = accuracy_score(val_df[LABEL_COLUMN], val_preds)
    val_macro_f1 = f1_score(val_df[LABEL_COLUMN], val_preds, average="macro")
    val_precision = precision_score(val_df[LABEL_COLUMN], val_preds, average="macro", zero_division=0)
    val_recall = recall_score(val_df[LABEL_COLUMN], val_preds, average="macro", zero_division=0)
    
    print(f"[{exp_code}] Hasil Validasi -> Acc: {val_accuracy:.4f} | Macro F1: {val_macro_f1:.4f}")
    
    return {
        "exp_code": exp_code,
        "use_ros": use_ros,
        "batch_size": batch_size,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_macro_f1,
        "val_precision": val_precision,
        "val_recall": val_recall
    }


In [ ]:
# 1. Definisikan Grid 4 Skenario Eksperimen
EXPERIMENT_CONFIGS = {
    "PB1": {"use_ros": False, "batch_size": 8},
    "PB2": {"use_ros": False, "batch_size": 16},
    "PB3": {"use_ros": True, "batch_size": 8},
    "PB4": {"use_ros": True, "batch_size": 16},
}

# 2. Jalankan Skenario
results = []
for exp_code, cfg in EXPERIMENT_CONFIGS.items():
    exp_res = run_single_split_experiment(exp_code, cfg["use_ros"], cfg["batch_size"])
    results.append(exp_res)
    
# 3. Ringkasan Hasil Eksperimen
comparison_df = pd.DataFrame(results)
comparison_df.to_csv(os.path.join(BASE_OUTPUT_DIR, "validation_comparison.csv"), index=False)

print("\n" + "="*80)
print("TABEL RINGKASAN HASIL EKSPERIMEN (SET VALIDASI)")
print("="*80)
display(comparison_df)

# 4. Pilih Konfigurasi Terbaik (berdasarkan Macro F1 tertinggi di set validasi)
best_scenario_idx = comparison_df["val_macro_f1"].idxmax()
best_scenario_row = comparison_df.iloc[best_scenario_idx]
best_exp_code = best_scenario_row["exp_code"]

print(f"\n[INFO] Konfigurasi Terbaik terpilih secara otomatis: **{best_exp_code}**")
print(f"       (ROS={best_scenario_row['use_ros']}, Batch Size={best_scenario_row['batch_size']})")
print(f"       Validation Macro F1: {best_scenario_row['val_macro_f1']:.4f}")


In [ ]:
# =============================================================================
# 5-FOLD CROSS-VALIDATION UNTUK KONFIGURASI TERBAIK
# =============================================================================
print("="*80)
print(f"FITTING 5-FOLD CROSS-VALIDATION UNTUK KONFIGURASI TERBAIK: {best_exp_code}")
print("="*80)

best_use_ros = EXPERIMENT_CONFIGS[best_exp_code]["use_ros"]
best_batch_size = EXPERIMENT_CONFIGS[best_exp_code]["batch_size"]

cv_dir = os.path.join(BASE_OUTPUT_DIR, f"CV_{best_exp_code}")
os.makedirs(cv_dir, exist_ok=True)

# Gabungkan train_df dan val_df kembali menjadi data training untuk CV (90% data)
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
print(f"Train + Val Data Shape untuk CV: {train_val_df.shape}")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_rows = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_val_df[TEXT_COLUMN], train_val_df[LABEL_COLUMN]), start=1):
    print(f"\n--- CV {best_exp_code} | Fold {fold}/{N_SPLITS} ---")
    
    fold_train = train_val_df.iloc[tr_idx]
    fold_val = train_val_df.iloc[val_idx]
    
    # 1. Terapkan ROS hanya pada data latih fold (mencegah leakage!)
    if best_use_ros:
        ros = RandomOverSampler(random_state=RANDOM_STATE)
        X_res, y_res = ros.fit_resample(
            fold_train[TEXT_COLUMN].values.reshape(-1, 1),
            fold_train[LABEL_COLUMN]
        )
        fold_train_text = X_res.flatten()
        fold_train_label = y_res
    else:
        fold_train_text = fold_train[TEXT_COLUMN].values
        fold_train_label = fold_train[LABEL_COLUMN].values
        
    # 2. Fit Tokenizer hanya pada data latih fold
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
    tokenizer.fit_on_texts(fold_train_text)
    
    # 3. Tokenisasi dan Padding
    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(fold_train_text), maxlen=MAX_LEN, padding="post", truncating="post")
    X_val_seq = pad_sequences(tokenizer.texts_to_sequences(fold_val[TEXT_COLUMN].values), maxlen=MAX_LEN, padding="post", truncating="post")
    
    model = build_model()
    early_stop = EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    
    # 4. Latih Model Fold
    history = model.fit(
        X_train_seq, fold_train_label,
        validation_data=(X_val_seq, fold_val[LABEL_COLUMN].values),
        epochs=EPOCHS,
        batch_size=best_batch_size,
        callbacks=[early_stop],
        verbose=1
    )
    
    pd.DataFrame(history.history).to_csv(os.path.join(cv_dir, f"history_fold_{fold}.csv"), index=False)
    plot_history(history, os.path.join(cv_dir, f"history_fold_{fold}.png"), f"Loss Curve - Fold {fold}")
    
    # 5. Evaluasi Fold
    preds = np.argmax(model.predict(X_val_seq), axis=1)
    acc = accuracy_score(fold_val[LABEL_COLUMN], preds)
    macro_f1 = f1_score(fold_val[LABEL_COLUMN], preds, average="macro")
    macro_prec = precision_score(fold_val[LABEL_COLUMN], preds, average="macro", zero_division=0)
    macro_rec = recall_score(fold_val[LABEL_COLUMN], preds, average="macro", zero_division=0)
    
    fold_rows.append({
        "Fold": fold,
        "Accuracy": acc,
        "Macro_F1": macro_f1,
        "Macro_Precision": macro_prec,
        "Macro_Recall": macro_rec
    })
    
    report = classification_report(fold_val[LABEL_COLUMN], preds, output_dict=True, zero_division=0)
    pd.DataFrame(report).transpose().to_csv(os.path.join(cv_dir, f"fold_{fold}_classification_report.csv"))

# Rekap CV
fold_df = pd.DataFrame(fold_rows)
mean_row = pd.DataFrame([{
    "Fold": "Mean",
    "Accuracy": fold_df["Accuracy"].mean(),
    "Macro_F1": fold_df["Macro_F1"].mean(),
    "Macro_Precision": fold_df["Macro_Precision"].mean(),
    "Macro_Recall": fold_df["Macro_Recall"].mean()
}])
cv_folds = pd.concat([fold_df, mean_row], ignore_index=True)
cv_folds.to_csv(os.path.join(cv_dir, "cv_folds.csv"), index=False)

cv_summary = pd.DataFrame([{
    "Model": f"BiLSTM_{best_exp_code}_CV",
    "Mean_Accuracy": fold_df["Accuracy"].mean(),
    "Mean_Macro_F1": fold_df["Macro_F1"].mean(),
    "Mean_Macro_Precision": fold_df["Macro_Precision"].mean(),
    "Mean_Macro_Recall": fold_df["Macro_Recall"].mean(),
    "Std_Accuracy": fold_df["Accuracy"].std(),
    "Std_Macro_F1": fold_df["Macro_F1"].std()
}])
cv_summary.to_csv(os.path.join(cv_dir, "cv_summary.csv"), index=False)

print("\n" + "="*80)
print(f"TABEL REKAP CROSS-VALIDATION (5-FOLD) UNTUK KONFIGURASI {best_exp_code}")
print("="*80)
display(cv_folds)

# Cari fold terbaik berdasarkan Macro F1 dan plot kurva loss-nya
best_fold_idx = fold_df['Macro_F1'].idxmax()
best_fold_row = fold_df.iloc[best_fold_idx]
best_fold_num = int(best_fold_row['Fold'])
print(f'\n[INFO] Fold terbaik: Fold {best_fold_num} (Macro F1: {best_fold_row["Macro_F1"]:.4f})')

best_fold_history = pd.read_csv(os.path.join(cv_dir, f'history_fold_{best_fold_num}.csv'))
plot_history(
    best_fold_history,
    os.path.join(cv_dir, f'best_fold_{best_fold_num}_loss_curve.png'),
    f'Loss Curve - Fold Terbaik (Fold {best_fold_num})'
)


In [ ]:
# =============================================================================
# PELATIHAN MODEL AKHIR & EVALUASI PADA TEST SET (10% DATA)
# =============================================================================
print("="*80)
print(f"PELATIHAN MODEL AKHIR MENGGUNAKAN KONFIGURASI TERBAIK: {best_exp_code}")
print("="*80)

final_dir = os.path.join(BASE_OUTPUT_DIR, f"Final_Model_{best_exp_code}")
os.makedirs(final_dir, exist_ok=True)

# 1. Gunakan split latih manual dari train_val_df (90% data) untuk melatih model akhir
#    Kita pisahkan 10% dari train_val_df untuk validasi early stopping secara internal
train_final, val_final = train_test_split(
    train_val_df, test_size=0.1, stratify=train_val_df[LABEL_COLUMN], random_state=RANDOM_STATE
)

# 2. Terapkan ROS hanya pada data latih final (jika diaktifkan)
if best_use_ros:
    ros_final = RandomOverSampler(random_state=RANDOM_STATE)
    X_res, y_res = ros_final.fit_resample(
        train_final[TEXT_COLUMN].values.reshape(-1, 1),
        train_final[LABEL_COLUMN]
    )
    train_final_text = X_res.flatten()
    train_final_label = y_res
else:
    train_final_text = train_final[TEXT_COLUMN].values
    train_final_label = train_final[LABEL_COLUMN].values

# 3. Fit Tokenizer akhir hanya pada data latih final
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_final_text)

with open(os.path.join(final_dir, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

# 4. Tokenisasi dan Padding
X_train_final = pad_sequences(tokenizer.texts_to_sequences(train_final_text), maxlen=MAX_LEN, padding="post", truncating="post")
X_val_final = pad_sequences(tokenizer.texts_to_sequences(val_final[TEXT_COLUMN].values), maxlen=MAX_LEN, padding="post", truncating="post")
X_test_final = pad_sequences(tokenizer.texts_to_sequences(test_df[TEXT_COLUMN].values), maxlen=MAX_LEN, padding="post", truncating="post")

# 5. Latih Model Akhir
final_model = build_model()
early_stop = EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)

final_model.fit(
    X_train_final, train_final_label,
    validation_data=(X_val_final, val_final[LABEL_COLUMN].values),
    epochs=EPOCHS,
    batch_size=best_batch_size,
    callbacks=[early_stop],
    verbose=1
)

final_model.save(os.path.join(final_dir, "bilstm_model.keras"))
print("[INFO] Model final berhasil disimpan!")

# =============================================================================
# EVALUASI UTAMA PADA TEST SET BERSIH
# =============================================================================
test_preds = np.argmax(final_model.predict(X_test_final), axis=1)

test_metrics = pd.DataFrame([{
    "Model": f"BiLSTM_{best_exp_code}",
    "Accuracy": accuracy_score(test_df[LABEL_COLUMN], test_preds),
    "Macro_F1": f1_score(test_df[LABEL_COLUMN], test_preds, average="macro"),
    "Macro_Precision": precision_score(test_df[LABEL_COLUMN], test_preds, average="macro", zero_division=0),
    "Macro_Recall": recall_score(test_df[LABEL_COLUMN], test_preds, average="macro", zero_division=0)
}])
test_metrics.to_csv(os.path.join(final_dir, "final_test_metrics.csv"), index=False)

print("\n" + "="*80)
print(f"HASIL METRIK PENGUJIAN AKHIR (TEST SET) MENGGUNAKAN {best_exp_code}")
print("="*80)
display(test_metrics)

# Simpan Laporan Klasifikasi & Confusion Matrix Pengujian Akhir
report_test = classification_report(test_df[LABEL_COLUMN], test_preds, output_dict=True, zero_division=0)
pd.DataFrame(report_test).transpose().to_csv(os.path.join(final_dir, "classification_report_test.csv"))

save_confusion_matrix(
    test_df[LABEL_COLUMN], test_preds,
    os.path.join(final_dir, "confusion_matrix_test.png"),
    f"Confusion Matrix Test Set - BiLSTM {best_exp_code}"
)

print("\n[SELESAI] Seluruh alur eksperimen Bi-LSTM telah berhasil dijalankan.")
